1. Install libraries

In [ ]:
!pip install -q datasets transformers accelerate sentencepiece scikit-learn seaborn

In [ ]:
!pip install openai

2. Import libraries

In [ ]:
import torch
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from datasets import load_dataset
from transformers import pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

3. Check GPU

In [ ]:
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

4. Load TweetEval sentiment dataset

In [ ]:
twitter = load_dataset("cardiffnlp/tweet_eval", "sentiment", trust_remote_code=True)

print(twitter)
print("Train labels:", set(twitter["train"]["label"]))
print("Test labels:", set(twitter["test"]["label"]))

5. Convert to binary sentiment

In [ ]:
# I remove neutral because the experiment compares positive and negative sentiment.

def convert_twitter_label(example):
    if example["label"] == 0:
        example["label"] = 0   # negative
    elif example["label"] == 2:
        example["label"] = 1   # positive
    else:
        example["label"] = -1  # neutral, remove later
    return example

twitter_binary = twitter.map(convert_twitter_label)
twitter_binary = twitter_binary.filter(lambda x: x["label"] != -1)

print("Binary train labels:", set(twitter_binary["train"]["label"]))
print("Binary test labels:", set(twitter_binary["test"]["label"]))
print(twitter_binary)

6. Select test samples

In [ ]:
# I keep 2000 samples like the D task.

TEST_SIZE = 2000

twitter_test = twitter_binary["test"].select(range(TEST_SIZE))

X_test_twitter = twitter_test["text"]
y_test_twitter = twitter_test["label"]

print("Twitter test size:", len(X_test_twitter))
print(twitter_test.column_names)
print("First tweet:", X_test_twitter[0])

Create Twitter special-case subsets

7. Define category rules

In [ ]:
# These rules are reused from the D task.

def is_short(text):
    return len(text.split()) < 8


def has_slang(text):
    slang_words = [
        "meh", "idk", "smh", "bruh", "ugh", "nah", "yikes",
        "lmao", "rofl", "wtf", "omg",
        "lowkey", "highkey", "deadass", "no cap", "cap",
        "sus", "trash", "fire", "mid",
        "this ain't it", "not it", "big yikes",
        "love that for you", "yeah right", "thanks a lot"
    ]
    text = text.lower()
    return any(word in text for word in slang_words)


def has_negation(text):
    text_lower = text.lower()
    negation_terms = [
        "not", "n't", "never", "no", "nothing", "hardly",
        "barely", "without", "neither", "nor"
    ]
    return any(term in text_lower.split() for term in negation_terms) or "n't" in text_lower


def has_mixed_sentiment(text):
    positive_words = [
        "good", "great", "nice", "love", "amazing",
        "best", "happy", "excellent", "perfect"
    ]
    negative_words = [
        "bad", "worst", "hate", "poor", "terrible",
        "awful", "sad", "trash", "disappointing"
    ]

    text = text.lower()
    return any(p in text for p in positive_words) and any(n in text for n in negative_words)


def has_uncertainty(text):
    uncertain_words = [
        "maybe", "perhaps", "i guess", "kind of",
        "sort of", "not sure", "it depends", "probably"
    ]

    text = text.lower()
    return any(word in text for word in uncertain_words)


def has_sarcasm_like(text):
    sarcasm_phrases = [
        "yeah right", "thanks a lot", "great...",
        "just perfect", "love that for you",
        "what a joke", "as if", "sure"
    ]

    text = text.lower()
    return any(phrase in text for phrase in sarcasm_phrases)


def has_emoji_or_symbol(text):
    emoji_symbols = [
        "😂", "😭", "😡", "😒", "❤️", "💀",
        "🔥", "🙄", "👍", "👎", "!"
    ]

    return any(symbol in text for symbol in emoji_symbols)

def is_ambiguous_negation(text):
    return (
        has_negation(text)
        or has_mixed_sentiment(text)
        or has_uncertainty(text)
        or has_sarcasm_like(text)
    )

8. Create balanced subsets

In [ ]:
#I balanced the subset for the fair comparision

import random

random.seed(42)

short_idx = [
    i for i, t in enumerate(X_test_twitter)
    if is_short(t)
]

slang_idx = [
    i for i, t in enumerate(X_test_twitter)
    if has_slang(t)
]

amb_idx = [
    i for i, t in enumerate(X_test_twitter)
    if is_ambiguous_negation(t)
]

print("Before balancing")
print("Short:", len(short_idx))
print("Slang:", len(slang_idx))
print("Ambiguous/Negation:", len(amb_idx))

BALANCED_SIZE = min(len(short_idx), len(slang_idx), len(amb_idx))

short_idx_bal = random.sample(short_idx, BALANCED_SIZE)
slang_idx_bal = random.sample(slang_idx, BALANCED_SIZE)
amb_idx_bal = random.sample(amb_idx, BALANCED_SIZE)

subset_indices = {
    "Short": short_idx_bal,
    "Slang": slang_idx_bal,
    "Ambiguous/Negation": amb_idx_bal
}


print("\nAfter balancing")
for name, idxs in subset_indices.items():
    print(name, len(idxs))

Shared evaluation functions

9. Metrics

In [ ]:
# I use the same metrics for every model.
# F1 is important because it balances precision and recall.

def calculate_metrics(y_true, y_pred):
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1-score": f1_score(y_true, y_pred, zero_division=0)
    }


def evaluate_overall(model_name, y_true, y_pred):
    row = {
        "Model": model_name,
        "Case": "Overall Twitter",
        "Samples": len(y_true)
    }
    row.update(calculate_metrics(y_true, y_pred))
    return row


def evaluate_subsets(model_name, y_all, pred_all, subset_indices):
    rows = []

    for case_name, indices in subset_indices.items():
        y_case = [y_all[i] for i in indices]
        pred_case = [pred_all[i] for i in indices]

        row = {
            "Model": model_name,
            "Case": case_name,
            "Samples": len(indices)
        }
        row.update(calculate_metrics(y_case, pred_case))
        rows.append(row)

    return rows


def label_to_text(label):
    return "positive" if label == 1 else "negative"

Model 1: Zero-shot DeBERTa

10. Load DeBERTa

In [ ]:
# Model 1 is the zero-shot DeBERTa model from the D task.

deberta_model_name = "cross-encoder/nli-deberta-v3-large"

deberta_classifier = pipeline(
    "zero-shot-classification",
    model=deberta_model_name,
    device=0 if torch.cuda.is_available() else -1
)

candidate_labels = ["negative", "positive"]

11. Predict with DeBERTa

In [ ]:
def predict_deberta(texts):
    preds = []

    for text in tqdm(texts):
        result = deberta_classifier(
            text,
            candidate_labels=candidate_labels,
            multi_label=False
        )

        best_label = result["labels"][0]
        preds.append(1 if best_label == "positive" else 0)

    return preds

In [ ]:
deberta_preds = predict_deberta(X_test_twitter)

deberta_results = [evaluate_overall("Zero-shot DeBERTa", y_test_twitter, deberta_preds)]
deberta_results += evaluate_subsets("Zero-shot DeBERTa", y_test_twitter, deberta_preds, subset_indices)

pd.DataFrame(deberta_results)

Model 2: Twitter-RoBERTa-latest

12. Load Twitter-RoBERTa

In [ ]:
# Model 2 is Twitter-specific.
# This checks if a model trained for tweets still has the same weak category.

twitter_roberta_model = "cardiffnlp/twitter-roberta-base-sentiment-latest"

twitter_roberta_classifier = pipeline(
    "text-classification",
    model=twitter_roberta_model,
    tokenizer=twitter_roberta_model,
    device=0 if torch.cuda.is_available() else -1,
    top_k=None
)

13. Predict with Twitter-RoBERTa

In [ ]:
def predict_twitter_roberta(texts):
    preds = []

    for text in tqdm(texts):
        output = twitter_roberta_classifier(text)[0]

        scores = {item["label"].lower(): item["score"] for item in output}

        neg_score = scores.get("negative", 0)
        pos_score = scores.get("positive", 0)

        preds.append(1 if pos_score >= neg_score else 0)

    return preds

In [ ]:
roberta_preds = predict_twitter_roberta(X_test_twitter)

roberta_results = [evaluate_overall("Twitter-RoBERTa-latest", y_test_twitter, roberta_preds)]
roberta_results += evaluate_subsets("Twitter-RoBERTa-latest", y_test_twitter, roberta_preds, subset_indices)

pd.DataFrame(roberta_results)

Model 3: Qwen3-4B

14. Load Qwen3-4B

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# Model 3: Qwen3-4B
# I use this as a recent open LLM comparison because GPT API has request limits.

qwen_model_name = "Qwen/Qwen3-4B"

qwen_tokenizer = AutoTokenizer.from_pretrained(
    qwen_model_name,
    trust_remote_code=True
)

qwen_model = AutoModelForCausalLM.from_pretrained(
    qwen_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

qwen_model.eval()

print("Qwen model loaded")

15. Qwen prediction function

In [ ]:
def predict_qwen_sentiment(texts, max_new_tokens=10):
    predictions = []

    for text in tqdm(texts):
        prompt = f"""
Classify the sentiment of this tweet as exactly one label: positive or negative.

Tweet:
{text}

Answer with one word only: positive or negative.
"""

        messages = [
            {"role": "user", "content": prompt}
        ]

        try:
            # Qwen3 supports enable_thinking=False in newer tokenizer versions
            input_text = qwen_tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False
            )
        except TypeError:
            input_text = qwen_tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True
            )

        inputs = qwen_tokenizer(
            input_text,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(qwen_model.device)

        with torch.no_grad():
            output_ids = qwen_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=qwen_tokenizer.eos_token_id
            )

        generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
        answer = qwen_tokenizer.decode(generated_ids, skip_special_tokens=True).strip().lower()

        if "positive" in answer and "negative" not in answer:
            predictions.append(1)
        elif "negative" in answer:
            predictions.append(0)
        else:
            # fallback if answer is unclear
            predictions.append(0)

    return predictions

16. Predict with Qwen model

In [ ]:
# I evaluate Qwen on the same 2,000 samples for fair overall comparison.

qwen_preds = predict_qwen_sentiment(X_test_twitter)

qwen_results = [evaluate_overall("Qwen3-4B zero-shot", y_test_twitter, qwen_preds)]

qwen_results += evaluate_subsets("Qwen3-4B zero-shot", y_test_twitter, qwen_preds, subset_indices)

pd.DataFrame(qwen_results)

Compare the three models

17. Model comparison table

In [ ]:
all_model_results = []

all_model_results += deberta_results
all_model_results += roberta_results
all_model_results += qwen_results

df_model_results = pd.DataFrame(all_model_results)

df_model_results

18. Focus on balanced special cases only

In [ ]:
df_special_comparison = df_model_results[
    df_model_results["Case"].isin(["Short", "Slang", "Ambiguous/Negation"])
].copy()

df_special_comparison

Error analysis for each model

19. Function to create error dataframe

In [ ]:
def create_error_dataframe(model_name, texts, y_true, y_pred):
    rows = []

    for i, (text, true_label, pred_label) in enumerate(zip(texts, y_true, y_pred)):
        if true_label != pred_label:
            rows.append({
                "model": model_name,
                "index": i,
                "text": text,
                "true_label": label_to_text(true_label),
                "predicted_label": label_to_text(pred_label)
            })

    df = pd.DataFrame(rows)

    if len(df) > 0:
        df["short"] = df["text"].apply(is_short)
        df["slang"] = df["text"].apply(has_slang)
        df["negation"] = df["text"].apply(has_negation)
        df["mixed_sentiment"] = df["text"].apply(has_mixed_sentiment)
        df["sarcasm_like"] = df["text"].apply(has_sarcasm_like)
        df["emoji_symbols"] = df["text"].apply(has_emoji_or_symbol)

    return df

20. Create DeBERTa and RoBERTa error dataframes

In [ ]:
df_deberta_errors = create_error_dataframe("Zero-shot DeBERTa",X_test_twitter,y_test_twitter,deberta_preds)

df_roberta_errors = create_error_dataframe("Twitter-RoBERTa-latest",X_test_twitter,y_test_twitter,roberta_preds)

print("DeBERTa errors:", len(df_deberta_errors))
print("RoBERTa errors:", len(df_roberta_errors))

21. Create Qwen error dataframe

In [ ]:
df_qwen_errors  = create_error_dataframe("Qwen3-4B zero-shot",X_test_twitter,y_test_twitter,qwen_preds)

print("Qwen errors:", len(df_qwen_errors))

22. False positive / false negative breakdown

In [ ]:
def false_error_breakdown(df_errors):
    if len(df_errors) == 0:
        return pd.DataFrame()

    false_negatives = df_errors[
        (df_errors["true_label"] == "positive") &
        (df_errors["predicted_label"] == "negative")
    ]

    false_positives = df_errors[
        (df_errors["true_label"] == "negative") &
        (df_errors["predicted_label"] == "positive")
    ]

    def count_patterns(error_df, error_type):
        return {
            "Model": df_errors["model"].iloc[0],
            "Error Type": error_type,
            "Total Errors": len(error_df),
            "Short": error_df["short"].sum(),
            "Slang": error_df["slang"].sum(),
            "Negation": error_df["negation"].sum(),
            "Mixed sentiment": error_df["mixed_sentiment"].sum(),
            "Sarcasm-like": error_df["sarcasm_like"].sum(),
            "Emoji/symbols": error_df["emoji_symbols"].sum()
        }

    return pd.DataFrame([
        count_patterns(false_negatives, "False Negative"),
        count_patterns(false_positives, "False Positive")
    ])

In [ ]:
breakdowns = []

breakdowns.append(false_error_breakdown(df_deberta_errors))
breakdowns.append(false_error_breakdown(df_roberta_errors))
breakdowns.append(false_error_breakdown(df_qwen_errors))

df_error_breakdown = pd.concat(breakdowns, ignore_index=True)

df_error_breakdown

Actual examples: correct and misleading cases

23. Create prediction comparison for balanced subsets

In [ ]:
# I create one table to inspect actual tweets.
# This helps show which model is correct or wrong for each example.

example_rows = []

for case_name, indices in subset_indices.items():
    for i in indices:
        correct_models = []
        wrong_models = []

        if deberta_preds[i] == y_test_twitter[i]:
            correct_models.append("DeBERTa")
        else:
            wrong_models.append("DeBERTa")

        if roberta_preds[i] == y_test_twitter[i]:
            correct_models.append("Twitter-RoBERTa")
        else:
            wrong_models.append("Twitter-RoBERTa")

        if qwen_preds[i] == y_test_twitter[i]:
            correct_models.append("Qwen3-4B")
        else:
            wrong_models.append("Qwen3-4B")

        row = {
            "case": case_name,
            "index": i,
            "text": X_test_twitter[i],
            "true_label": label_to_text(y_test_twitter[i]),

            "deberta_pred": label_to_text(deberta_preds[i]),
            "roberta_pred": label_to_text(roberta_preds[i]),
            "qwen_pred": label_to_text(qwen_preds[i]),

            "correct_models": ", ".join(correct_models) if correct_models else "None",
            "wrong_models": ", ".join(wrong_models) if wrong_models else "None",

            "short": is_short(X_test_twitter[i]),
            "slang": has_slang(X_test_twitter[i]),
            "negation": has_negation(X_test_twitter[i]),
            "mixed_sentiment": has_mixed_sentiment(X_test_twitter[i]),
            "sarcasm_like": has_sarcasm_like(X_test_twitter[i]),
            "emoji_symbols": has_emoji_or_symbol(X_test_twitter[i])
        }

        example_rows.append(row)

df_examples = pd.DataFrame(example_rows)

In [ ]:
df_examples[df_examples["case"] == "Short"].head(10)

In [ ]:
df_examples[df_examples["case"] == "Slang"].head(10)

In [ ]:
df_examples[df_examples["case"] == "Ambiguous/Negation"].head(10)

24. Show Correct examples

Samples of all 3 models worked correctly

In [ ]:
df_examples[
    (df_examples["case"] == "Short") &
    (df_examples["wrong_models"].str.contains("None", na=False))
].head(10)

In [ ]:
df_examples[
    (df_examples["case"] == "Slang") &
    (df_examples["wrong_models"].str.contains("None", na=False))
].head(10)

In [ ]:
df_examples[
    (df_examples["case"] == "Ambiguous/Negation") &
    (df_examples["wrong_models"].str.contains("None", na=False))
].head(10)

25. Show misleading examples

Samples od all 3 models worked wrongly

In [ ]:
df_examples[
    (df_examples["case"] == "Short") &
    (df_examples["correct_models"].str.contains("None", na=False))
].head(10)

In [ ]:
df_examples[
    (df_examples["case"] == "Slang") &
    (df_examples["correct_models"].str.contains("None", na=False))
].head(10)

In [ ]:
df_examples[
    (df_examples["case"] == "Ambiguous/Negation") &
    (df_examples["correct_models"].str.contains("None", na=False))
].head(10)

Other wrong case samples

In [ ]:
df_examples[
    (df_examples["case"] == "Short") &
    ~(df_examples["wrong_models"].str.contains("None", na=False))
].head(20)

In [ ]:
df_examples[
    (df_examples["case"] == "Slang") &
    ~(df_examples["wrong_models"].str.contains("None", na=False))
].head(20)

In [ ]:
df_examples[
    (df_examples["case"] == "Ambiguous/Negation") &
    ~(df_examples["wrong_models"].str.contains("None", na=False))
].head(10)

Confusion matrix function

In [ ]:
# I use the same confusion matrix function for all models.
# This makes the comparison easier and consistent.

import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix(y_true, y_pred, title, file_name):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Purples",
        xticklabels=["Negative", "Positive"],
        yticklabels=["Negative", "Positive"]
    )

    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(file_name, dpi=300)
    plt.show()

    return cm

DeBERTa confusion matrix

In [ ]:
cm_deberta = plot_confusion_matrix(
    y_test_twitter,
    deberta_preds,
    "Zero-shot DeBERTa Confusion Matrix",
    "confusion_matrix_deberta.png"
)

cm_deberta

Twitter-RoBERTa confusion matrix

In [ ]:
cm_roberta = plot_confusion_matrix(
    y_test_twitter,
    roberta_preds,
    "Twitter-RoBERTa Confusion Matrix",
    "confusion_matrix_twitter_roberta.png"
)

cm_roberta

Qwen3-4B confusion matrix

In [ ]:
cm_qwen = plot_confusion_matrix(
    y_test_twitter,
    qwen_preds,
    "Qwen3-4B Confusion Matrix",
    "confusion_matrix_qwen3_4b.png"
)

cm_qwen

**Proposed solution**: Context-Aware Sentiment Normalisation

It is not just “try another model”. It changes the input for difficult ambiguous tweets.

Separate model for the proposed solution

In [ ]:
# Separate model only for the proposed normalisation method.
# I keep this separate so my Qwen classifier baseline does not change.

normaliser_model_name = "Qwen/Qwen3-4B-Instruct-2507"

normaliser_tokenizer = AutoTokenizer.from_pretrained(
    normaliser_model_name,
    trust_remote_code=True
)

normaliser_model = AutoModelForCausalLM.from_pretrained(
    normaliser_model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

normaliser_model.eval()

print("Normalisation model loaded:", normaliser_model_name)

30. Normalisation prompt

In [ ]:
# Proposed solution:

# I normalise ambiguous/negation tweets into clearer sentiment wording using Qwen.
# This avoids using GPT API and keeps the experiment reproducible in Colab.

def normalise_tweet_with_qwen_instruct(text, max_new_tokens=60):
    prompt = f"""
You are preparing a tweet for sentiment classification.

Do not rewrite the tweet completely.
Do not change the sentiment.
Do not add new facts.
Keep important clues such as emojis, slang, hashtags, negation, and sarcasm.

Write a short clarification of the tweet's sentiment in plain English.
The clarification should help a classifier decide whether the tweet is positive or negative.

Tweet:
{text}

Return only this format:
Sentiment clarification: <short clarification>
"""

    messages = [
        {"role": "user", "content": prompt}
    ]

    try:
        input_text = normaliser_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False
        )
    except TypeError:
        input_text = normaliser_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

    inputs = normaliser_tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(normaliser_model.device)

    with torch.no_grad():
        output_ids = normaliser_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=normaliser_tokenizer.eos_token_id
        )

    generated_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
    clarification = normaliser_tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

    return clarification

31. Normalise ambiguous/negation tweets

In [ ]:
# I only normalise the ambiguous/negation subset because that is the weak case.
#Combine original tweet + clarification:

normalised_rows = []

for i in tqdm(amb_idx_bal):
    original_text = X_test_twitter[i]
    clarification = normalise_tweet_with_qwen_instruct(original_text)

    combined_text = (
        "Original tweet: " + original_text +
        "\nAdditional sentiment clarification: " + clarification
    )

    normalised_rows.append({
        "index": i,
        "original_text": original_text,
        "clarification": clarification,
        "combined_text": combined_text,
        "true_label": y_test_twitter[i]
    })

df_normalised = pd.DataFrame(normalised_rows)
df_normalised.head(10)

In [ ]:
X_norm = list(df_normalised["combined_text"])
y_norm = list(df_normalised["true_label"])

32. Classify normalised text using DeBERTa

In [ ]:
deberta_norm_preds =predict_deberta(X_norm)

deberta_norm_result = {
    "Model": "DeBERTa + Qwen Normalisation",
    "Case": "Ambiguous/Negation",
    "Samples": len(y_norm)
}
deberta_norm_result.update(calculate_metrics(y_norm, deberta_norm_preds))

pd.DataFrame([deberta_norm_result])

33. Classify normalised text using Twitter-RoBERTa

In [ ]:
roberta_norm_preds = predict_twitter_roberta(X_norm)

roberta_norm_result = {
    "Model": "Twitter-RoBERTa + Qwen Normalisation",
    "Case": "Ambiguous/Negation",
    "Samples": len(y_norm)
}
roberta_norm_result.update(calculate_metrics(y_norm, roberta_norm_preds))

pd.DataFrame([roberta_norm_result])

33A. Classify normalised text using Qwen

In [ ]:
qwen_norm_preds = predict_qwen_sentiment(X_norm)

qwen_norm_result = {
    "Model": "Qwen3-4B + Normalisation",
    "Case": "Ambiguous/Negation",
    "Samples": len(y_norm)
}
qwen_norm_result.update(calculate_metrics(y_norm, qwen_norm_preds))

pd.DataFrame([qwen_norm_result])

34. Compare before and after normalisation

In [ ]:
if len(df_normalised) > 0:
    before_after_rows = []

    # DeBERTa before
    before_after_rows.append(
        df_special_comparison[
            (df_special_comparison["Model"] == "Zero-shot DeBERTa") &
            (df_special_comparison["Case"] == "Ambiguous/Negation")
        ].iloc[0].to_dict()
    )

    # DeBERTa after
    before_after_rows.append(deberta_norm_result)

    # Twitter-RoBERTa before
    before_after_rows.append(
        df_special_comparison[
            (df_special_comparison["Model"] == "Twitter-RoBERTa-latest") &
            (df_special_comparison["Case"] == "Ambiguous/Negation")
        ].iloc[0].to_dict()
    )

    # Twitter-RoBERTa after
    before_after_rows.append(roberta_norm_result)

    # GPT before
    before_after_rows.append(
        df_special_comparison[
            (df_special_comparison["Model"] == "Qwen3-4B zero-shot") &
            (df_special_comparison["Case"] == "Ambiguous/Negation")
        ].iloc[0].to_dict()
    )

    # GPT after
    before_after_rows.append(qwen_norm_result)

    df_before_after = pd.DataFrame(before_after_rows)

    df_before_after[
        ["Model", "Case", "Samples", "Accuracy", "Precision", "Recall", "F1-score"]
    ]

    display(df_before_after)

else:
    df_before_after = pd.DataFrame()
    print("No normalised data available.")

Create error dataframe for normalised results

In [ ]:
# I do the same error analysis for the proposed method.
# This checks what errors remain after normalisation.

def create_normalised_error_dataframe(model_name, texts, y_true, y_pred):
    rows = []

    for i, (text, true_label, pred_label) in enumerate(zip(texts, y_true, y_pred)):
        if true_label != pred_label:
            rows.append({
                "model": model_name,
                "index": i,
                "text": text,
                "true_label": label_to_text(true_label),
                "predicted_label": label_to_text(pred_label)
            })

    df = pd.DataFrame(rows)

    if len(df) > 0:
        df["short"] = df["text"].apply(is_short)
        df["slang"] = df["text"].apply(has_slang)
        df["negation"] = df["text"].apply(has_negation)
        df["mixed_sentiment"] = df["text"].apply(has_mixed_sentiment)
        df["sarcasm_like"] = df["text"].apply(has_sarcasm_like)
        df["emoji_symbols"] = df["text"].apply(has_emoji_or_symbol)

    return df

Create normalised error dataframes for all 3 models

In [ ]:
df_deberta_norm_errors = create_normalised_error_dataframe(
    "DeBERTa + Normalisation",
    X_norm,
    y_norm,
    deberta_norm_preds
)

df_roberta_norm_errors = create_normalised_error_dataframe(
    "Twitter-RoBERTa + Normalisation",
    X_norm,
    y_norm,
    roberta_norm_preds
)

df_qwen_norm_errors = create_normalised_error_dataframe(
    "Qwen3-4B + Normalisation",
    X_norm,
    y_norm,
    qwen_norm_preds
)

print("DeBERTa normalised errors:", len(df_deberta_norm_errors))
print("Twitter-RoBERTa normalised errors:", len(df_roberta_norm_errors))
print("Qwen normalised errors:", len(df_qwen_norm_errors))

Count error categories after normalisation

In [ ]:
def error_pattern_summary(df_errors, model_name):
    return {
        "Model": model_name,
        "Total Errors": len(df_errors),
        "Short": df_errors["short"].sum() if len(df_errors) > 0 else 0,
        "Slang": df_errors["slang"].sum() if len(df_errors) > 0 else 0,
        "Negation": df_errors["negation"].sum() if len(df_errors) > 0 else 0,
        "Mixed sentiment": df_errors["mixed_sentiment"].sum() if len(df_errors) > 0 else 0,
        "Sarcasm-like": df_errors["sarcasm_like"].sum() if len(df_errors) > 0 else 0,
        "Emoji/symbols": df_errors["emoji_symbols"].sum() if len(df_errors) > 0 else 0
    }

df_norm_error_summary = pd.DataFrame([
    error_pattern_summary(df_deberta_norm_errors, "DeBERTa + Normalisation"),
    error_pattern_summary(df_roberta_norm_errors, "Twitter-RoBERTa + Normalisation"),
    error_pattern_summary(df_qwen_norm_errors, "Qwen3-4B + Normalisation")
])

df_norm_error_summary

False positive / false negative breakdown after normalisation

In [ ]:
df_norm_fp_fn_breakdown = pd.concat(
    [
        false_error_breakdown(df_deberta_norm_errors),
        false_error_breakdown(df_roberta_norm_errors),
        false_error_breakdown(df_qwen_norm_errors)
    ],
    ignore_index=True
)

df_norm_fp_fn_breakdown

Normalised examples table

In [ ]:
# I create the same style table for the proposed method.
# This keeps the analysis consistent with the baseline model comparison.

normalised_example_rows = []

for j in range(len(X_norm)):
    true_label = y_norm[j]

    correct_models = []
    wrong_models = []

    if deberta_norm_preds[j] == true_label:
        correct_models.append("DeBERTa+Norm")
    else:
        wrong_models.append("DeBERTa+Norm")

    if roberta_norm_preds[j] == true_label:
        correct_models.append("Twitter-RoBERTa+Norm")
    else:
        wrong_models.append("Twitter-RoBERTa+Norm")

    if qwen_norm_preds[j] == true_label:
        correct_models.append("Qwen3-4B+Norm")
    else:
        wrong_models.append("Qwen3-4B+Norm")

    row = {
        "case": "Ambiguous/Negation",
        "index": df_normalised.iloc[j]["index"],
        "original_text": df_normalised.iloc[j]["original_text"],
        "clarification": df_normalised.iloc[j].get("clarification", ""),
        "normalised_text": X_norm[j],
        "true_label": label_to_text(true_label),

        "deberta_norm_pred": label_to_text(deberta_norm_preds[j]),
        "roberta_norm_pred": label_to_text(roberta_norm_preds[j]),
        "qwen_norm_pred": label_to_text(qwen_norm_preds[j]),

        "correct_models": ", ".join(correct_models) if correct_models else "None",
        "wrong_models": ", ".join(wrong_models) if wrong_models else "None",

        "short": is_short(df_normalised.iloc[j]["original_text"]),
        "slang": has_slang(df_normalised.iloc[j]["original_text"]),
        "negation": has_negation(df_normalised.iloc[j]["original_text"]),
        "mixed_sentiment": has_mixed_sentiment(df_normalised.iloc[j]["original_text"]),
        "sarcasm_like": has_sarcasm_like(df_normalised.iloc[j]["original_text"]),
        "emoji_symbols": has_emoji_or_symbol(df_normalised.iloc[j]["original_text"])
    }

    normalised_example_rows.append(row)

df_norm_examples = pd.DataFrame(normalised_example_rows)

df_norm_examples.head(10)

Show normalised examples where all models are correct

In [ ]:
df_norm_examples[
    df_norm_examples["wrong_models"] == "None"
][
    [
        "original_text", "clarification", "true_label",
        "deberta_norm_pred", "roberta_norm_pred", "qwen_norm_pred","short","slang","negation","mixed_sentiment","sarcasm_like","emoji_symbols"
    ]
].head(10)

Show normalised examples where all 3 models are wrong

In [ ]:
df_norm_examples[
    df_norm_examples["correct_models"] == "None"
][
    [
        "original_text", "clarification", "true_label",
        "deberta_norm_pred", "roberta_norm_pred", "qwen_norm_pred",
        "short","slang","negation","mixed_sentiment","sarcasm_like","emoji_symbols"
    ]
].head(10)

Othe worng samples

In [ ]:
df_norm_examples[
    (df_norm_examples["wrong_models"] != "None") & (df_norm_examples["correct_models"] != "None")
][
    [
        "original_text", "clarification", "true_label",
        "deberta_norm_pred", "roberta_norm_pred", "qwen_norm_pred",
        "correct_models","wrong_models",
        "short","slang","negation","mixed_sentiment","sarcasm_like","emoji_symbols"
    ]
].head(10)

Did normalisation fix actual mistakes?

DeBERTa: fixed mistakes

In [ ]:
deberta_amb_original_preds = [deberta_preds[i] for i in amb_idx_bal]

df_norm_examples["deberta_original_pred"] = [label_to_text(p) for p in deberta_amb_original_preds]
df_norm_examples["deberta_original_correct"] = [
    p == y for p, y in zip(deberta_amb_original_preds, y_norm)
]
df_norm_examples["deberta_norm_correct"] = [
    p == y for p, y in zip(deberta_norm_preds, y_norm)
]

df_deberta_fixed = df_norm_examples[
    (df_norm_examples["deberta_original_correct"] == False) &
    (df_norm_examples["deberta_norm_correct"] == True)
]

print("DeBERTa mistakes fixed by normalisation:", len(df_deberta_fixed))

df_deberta_fixed[
    [
        "original_text", "clarification", "true_label",
        "deberta_original_pred", "deberta_norm_pred",
        "short","slang","negation","mixed_sentiment","sarcasm_like","emoji_symbols"
    ]
].head(10)

Twitter-RoBERTa: fixed mistakes

In [ ]:
roberta_amb_original_preds = [roberta_preds[i] for i in amb_idx_bal]

df_norm_examples["roberta_original_pred"] = [label_to_text(p) for p in roberta_amb_original_preds]
df_norm_examples["roberta_original_correct"] = [
    p == y for p, y in zip(roberta_amb_original_preds, y_norm)
]
df_norm_examples["roberta_norm_correct"] = [
    p == y for p, y in zip(roberta_norm_preds, y_norm)
]

df_roberta_fixed = df_norm_examples[
    (df_norm_examples["roberta_original_correct"] == False) &
    (df_norm_examples["roberta_norm_correct"] == True)
]

print("Twitter-RoBERTa mistakes fixed by normalisation:", len(df_roberta_fixed))

df_roberta_fixed[
    [
        "original_text", "clarification", "true_label",
        "roberta_original_pred", "roberta_norm_pred",
        "short","slang","negation","mixed_sentiment","sarcasm_like","emoji_symbols"
    ]
].head(10)

Qwen: fixed mistakes

In [ ]:
qwen_amb_original_preds = [qwen_preds[i] for i in amb_idx_bal]

df_norm_examples["qwen_original_pred"] = [label_to_text(p) for p in qwen_amb_original_preds]
df_norm_examples["qwen_original_correct"] = [
    p == y for p, y in zip(qwen_amb_original_preds, y_norm)
]
df_norm_examples["qwen_norm_correct"] = [
    p == y for p, y in zip(qwen_norm_preds, y_norm)
]

df_qwen_fixed = df_norm_examples[
    (df_norm_examples["qwen_original_correct"] == False) &
    (df_norm_examples["qwen_norm_correct"] == True)
]

print("Qwen mistakes fixed by normalisation:", len(df_qwen_fixed))

df_qwen_fixed[
    [
        "original_text", "clarification", "true_label",
        "qwen_original_pred", "qwen_norm_pred",
        "short","slang","negation","mixed_sentiment","sarcasm_like","emoji_symbols"
    ]
].head(10)

Also check mistakes introduced by normalisation

In [ ]:
df_deberta_worse = df_norm_examples[
    (df_norm_examples["deberta_original_correct"] == True) &
    (df_norm_examples["deberta_norm_correct"] == False)
]

df_roberta_worse = df_norm_examples[
    (df_norm_examples["roberta_original_correct"] == True) &
    (df_norm_examples["roberta_norm_correct"] == False)
]

df_qwen_worse = df_norm_examples[
    (df_norm_examples["qwen_original_correct"] == True) &
    (df_norm_examples["qwen_norm_correct"] == False)
]

print("DeBERTa made worse:", len(df_deberta_worse))
print("Twitter-RoBERTa made worse:", len(df_roberta_worse))
print("Qwen made worse:", len(df_qwen_worse))

Ambiguous/Negation subset errors before and after normalisation

In [ ]:
# Original DeBERTa predictions on the ambiguous/negation subset
deberta_amb_original_preds = [deberta_preds[i] for i in amb_idx_bal]
y_amb = [y_test_twitter[i] for i in amb_idx_bal]

# Count original errors before normalisation
deberta_amb_before_errors = sum(
    pred != true for pred, true in zip(deberta_amb_original_preds, y_amb)
)

# Count errors after normalisation
deberta_amb_after_errors = sum(
    pred != true for pred, true in zip(deberta_norm_preds, y_norm)
)

print("DeBERTa ambiguous/negation errors before normalisation:", deberta_amb_before_errors)
print("DeBERTa ambiguous/negation errors after normalisation:", deberta_amb_after_errors)
print("Error reduction:", deberta_amb_before_errors - deberta_amb_after_errors)

In [ ]:
# Labels for ambiguous/negation subset
y_amb = [y_test_twitter[i] for i in amb_idx_bal]

# Original predictions on ambiguous/negation subset
deberta_amb_original_preds = [deberta_preds[i] for i in amb_idx_bal]
roberta_amb_original_preds = [roberta_preds[i] for i in amb_idx_bal]
qwen_amb_original_preds = [qwen_preds[i] for i in amb_idx_bal]

# Count before and after errors
amb_error_before_after = pd.DataFrame([
    {
        "Model": "DeBERTa",
        "Before errors": sum(p != y for p, y in zip(deberta_amb_original_preds, y_amb)),
        "After errors": sum(p != y for p, y in zip(deberta_norm_preds, y_norm))
    },
    {
        "Model": "Twitter-RoBERTa",
        "Before errors": sum(p != y for p, y in zip(roberta_amb_original_preds, y_amb)),
        "After errors": sum(p != y for p, y in zip(roberta_norm_preds, y_norm))
    },
    {
        "Model": "Qwen3-4B",
        "Before errors": sum(p != y for p, y in zip(qwen_amb_original_preds, y_amb)),
        "After errors": sum(p != y for p, y in zip(qwen_norm_preds, y_norm))
    }
])

amb_error_before_after["Error reduction"] = (
    amb_error_before_after["Before errors"] -
    amb_error_before_after["After errors"]
)

amb_error_before_after

Confusion matrices

In [ ]:
# Ambiguous/Negation subset labels and original predictions
y_amb = [y_test_twitter[i] for i in amb_idx_bal]

deberta_amb_original_preds = [deberta_preds[i] for i in amb_idx_bal]
roberta_amb_original_preds = [roberta_preds[i] for i in amb_idx_bal]
qwen_amb_original_preds = [qwen_preds[i] for i in amb_idx_bal]

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

def plot_subset_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(4, 3))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Purples",
        xticklabels=["Negative", "Positive"],
        yticklabels=["Negative", "Positive"]
    )

    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.tight_layout()
    plt.show()

    return cm

DeBERTa before and after

In [ ]:
cm_deberta_amb_before = plot_subset_confusion_matrix(
    y_amb,
    deberta_amb_original_preds,
    "DeBERTa Before Normalisation - Ambiguous/Negation"
)

cm_deberta_amb_after = plot_subset_confusion_matrix(
    y_norm,
    deberta_norm_preds,
    "DeBERTa After Normalisation - Ambiguous/Negation"
)

Twitter-RoBERTa before and after

In [ ]:
cm_roberta_amb_before = plot_subset_confusion_matrix(
    y_amb,
    roberta_amb_original_preds,
    "Twitter-RoBERTa Before Normalisation - Ambiguous/Negation"
)

cm_roberta_amb_after = plot_subset_confusion_matrix(
    y_norm,
    roberta_norm_preds,
    "Twitter-RoBERTa After Normalisation - Ambiguous/Negation"
)

Qwen before and after

In [ ]:
cm_qwen_amb_before = plot_subset_confusion_matrix(
    y_amb,
    qwen_amb_original_preds,
    "Qwen3-4B Before Normalisation - Ambiguous/Negation"
)

cm_qwen_amb_after = plot_subset_confusion_matrix(
    y_norm,
    qwen_norm_preds,
    "Qwen3-4B After Normalisation - Ambiguous/Negation"
)

In [ ]:
normalised_cm_summary = pd.DataFrame([
    {
        "Model": "DeBERTa Before",
        "TN": cm_deberta_amb_before[0][0],
        "FP": cm_deberta_amb_before[0][1],
        "FN": cm_deberta_amb_before[1][0],
        "TP": cm_deberta_amb_before[1][1]
    },
    {
        "Model": "DeBERTa After",
        "TN": cm_deberta_amb_after[0][0],
        "FP": cm_deberta_amb_after[0][1],
        "FN": cm_deberta_amb_after[1][0],
        "TP": cm_deberta_amb_after[1][1]
    },
    {
        "Model": "RoBERTa Before",
        "TN": cm_roberta_amb_before[0][0],
        "FP": cm_roberta_amb_before[0][1],
        "FN": cm_roberta_amb_before[1][0],
        "TP": cm_roberta_amb_before[1][1]
    },
    {
        "Model": "RoBERTa After",
        "TN": cm_roberta_amb_after[0][0],
        "FP": cm_roberta_amb_after[0][1],
        "FN": cm_roberta_amb_after[1][0],
        "TP": cm_roberta_amb_after[1][1]
    },
    {
        "Model": "Qwen Before",
        "TN": cm_qwen_amb_before[0][0],
        "FP": cm_qwen_amb_before[0][1],
        "FN": cm_qwen_amb_before[1][0],
        "TP": cm_qwen_amb_before[1][1]
    },
    {
        "Model": "Qwen After",
        "TN": cm_qwen_amb_after[0][0],
        "FP": cm_qwen_amb_after[0][1],
        "FN": cm_qwen_amb_after[1][0],
        "TP": cm_qwen_amb_after[1][1]
    }
])

normalised_cm_summary